In [23]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from einops import rearrange, repeat

class ChannelAttention3D(nn.Module):
    def __init__(self, in_channels, reduction_ratio=16):
        """
        Channel attention module for 3D data
        
        Args:
            in_channels: Number of input channels
            reduction_ratio: Reduction ratio for the MLP
        """
        super(ChannelAttention3D, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool3d(1)
        self.max_pool = nn.AdaptiveMaxPool3d(1)
        
        # Shared MLP for both pooled features
        self.mlp = nn.Sequential(
            nn.Conv3d(in_channels, in_channels // reduction_ratio, kernel_size=1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv3d(in_channels // reduction_ratio, in_channels, kernel_size=1, bias=False)
        )
        
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        # Apply average pooling and max pooling
        avg_out = self.mlp(self.avg_pool(x))
        max_out = self.mlp(self.max_pool(x))
        
        # Combine the features and apply sigmoid activation
        out = self.sigmoid(avg_out + max_out)
        
        return out


class SpatialAttention3D(nn.Module):
    def __init__(self, kernel_size=7):
        """
        Spatial attention module for 3D data
        
        Args:
            kernel_size: Size of the convolutional kernel
        """
        super(SpatialAttention3D, self).__init__()
        
        assert kernel_size in (3, 5, 7), "Kernel size must be 3, 5, or 7"
        padding = kernel_size // 2
        
        self.conv = nn.Conv3d(2, 1, kernel_size=(kernel_size, kernel_size, kernel_size), 
                             padding=(padding, padding, padding), bias=False)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        # Apply average pooling and max pooling along channel dimension
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        
        # Concatenate the features
        out = torch.cat([avg_out, max_out], dim=1)
        
        # Apply convolution and sigmoid activation
        out = self.conv(out)
        out = self.sigmoid(out)
        
        return out


class CBAM3D(nn.Module):
    def __init__(self, in_channels, reduction_ratio=2, spatial_kernel_size=3):
        """
        Convolutional Block Attention Module (CBAM) for 3D data
        
        Args:
            in_channels: Number of input channels
            reduction_ratio: Reduction ratio for the channel attention MLP
            spatial_kernel_size: Kernel size for the spatial attention convolution
        """
        super(CBAM3D, self).__init__()
        
        self.channel_attention = ChannelAttention3D(in_channels, reduction_ratio)
        self.spatial_attention = SpatialAttention3D(spatial_kernel_size)
    
    def forward(self, x):
        # Store the input for the skip connection
        identity = x
        # print('in cbam: ',x.shape)
        # Apply channel attention
        x = x * self.channel_attention(x)
        # print('in cbam: ',x.shape)
        # Apply spatial attention
        x = x * self.spatial_attention(x)
        # print('in cbam: ',x.shape)
        # Add skip connection
        x = x + identity
        
        return x


# Example usage
class ResBlock3D_with_CBAM(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1):
        super(ResBlock3D_with_CBAM, self).__init__()
        
        # Main path
        self.conv1 = nn.Conv3d(in_channels, out_channels//2, kernel_size=kernel_size, stride=stride, padding=0, bias=False)
        self.bn1 = nn.BatchNorm3d(out_channels//2)
        self.relu = nn.ReLU(inplace=True)
        
        self.conv2 = nn.Conv3d(out_channels//2, out_channels, kernel_size=kernel_size, padding=0, bias=False)
        self.bn2 = nn.BatchNorm3d(out_channels)
        
        # CBAM attention module
        self.cbam = CBAM3D(out_channels)
        
        # Skip connection
        # self.skip = nn.Sequential()
        # if stride != 1 or in_channels != out_channels:
            # self.skip = nn.Sequential(
            #     nn.Conv3d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
            #     nn.BatchNorm3d(out_channels)
            # )
        self.skip = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=1, stride=2, bias=False),
            nn.BatchNorm3d(out_channels)
        )


    def forward(self, x):
        # Store input for skip connection
        identity = x
        
        # Main path
        out = self.conv1(x)
        # print('first in shape:', x.shape, ', first out shape: ', out.shape)
        out = self.bn1(out)
        out = self.relu(out)
        
        # print('second in shape:', out.shape)
        out = self.conv2(out) 
        # print('second out shape:', out.shape)       
        out = self.bn2(out)
        
        # Apply CBAM
        out = self.cbam(out)
        # print(out.shape, x.shape)
        # Add skip connection
        out += self.skip(identity)
        
        # Final activation
        out = self.relu(out)
        
        return out

class PatchEmbedding3D(nn.Module):
    def __init__(self, in_channels, embed_dim, patch_size):
        super().__init__()
        # self.proj = nn.Conv3d(in_channels, embed_dim, 
                            #   kernel_size=patch_size, stride=patch_size)

        self.proj = ResBlock3D_with_CBAM(in_channels, embed_dim, 
                              kernel_size=3, stride=patch_size)

    def forward(self, x):
        # x: (B, C, D1, D2, D3)
        x = self.proj(x)  # (B, embed_dim, D1//patch_size, D2//patch_size, D3//patch_size)
        x = rearrange(x, 'b c d1 d2 d3 -> b (d1 d2 d3) c')  # (B, N_patches, embed_dim)
        return x

class MultiHeadAttention(nn.Module):
    def __init__(self, dim, num_heads, dropout=0.0):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]  # (B, num_heads, N, head_dim)
        
        attn = (q @ k.transpose(-2, -1)) * self.scale  # (B, num_heads, N, N)
        attn = attn.softmax(dim=-1)
        attn = self.dropout(attn)
        
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)  # (B, N, C)
        x = self.proj(x)
        x = self.dropout(x)
        return x

class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim),
            nn.Dropout(dropout)
        )
    
    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = MultiHeadAttention(dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(dim)
        self.ffn = FeedForward(dim, int(dim * mlp_ratio), dropout)
        
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

class Vision3DTransformer(nn.Module):
    def __init__(
        self,
        in_channels,
        patch_size,
        embed_dim,
        depth,
        num_heads,
        mlp_ratio=4,
        dropout=0.0,
        cross_attn_dim=None,  # Dimension for cross-attention with LLM
    ):
        super().__init__()
        self.patch_embed = PatchEmbedding3D(in_channels, embed_dim, patch_size)
        
        # Position embedding
        # self.pos_embed = nn.Parameter(torch.zeros(1, 1000, embed_dim))  # Max 1000 patches as placeholder
        
        # Transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio, dropout)
            for _ in range(depth)
        ])
        
        self.norm = nn.LayerNorm(embed_dim)
        
        # Project to cross-attention dimension if needed
        self.cross_attn_dim = cross_attn_dim
        if cross_attn_dim is not None and cross_attn_dim != embed_dim:
            self.cross_attn_proj = nn.Linear(embed_dim, cross_attn_dim)
        else:
            self.cross_attn_proj = nn.Identity()
            
        # Initialize weights
        # self._init_weights()
    
    # def _init_weights(self):
        # Initialize position embeddings
        # nn.init.normal_(self.pos_embed, std=0.02)
    
    def forward(self, x):
        # x: (B, C, D1, D2, D3)
        B = x.shape[0]
        
        # Patch embeddings
        x = self.patch_embed(x)  # (B, N_patches, embed_dim)
        N_patches = x.shape[1]
        
        # Add position embeddings
        # pos_embed = self.pos_embed[:, :N_patches, :]
        # x = x + pos_embed
        
        # Apply transformer blocks
        for block in self.blocks:
            x = block(x)
        
        # Apply final normalization
        x = self.norm(x)
        
        # Project to cross-attention dimension if needed
        x = self.cross_attn_proj(x)
        
        return x  # (B, N_patches, cross_attn_dim or embed_dim)

# Example usage
def create_3d_vit(n_samp, n_channels, dim1, dim2, dim3, patch_size=1, embed_dim=128, depth=2, num_heads=4, mlp_ratio=2, cross_attn_dim=256):
    model = Vision3DTransformer(
        in_channels=n_channels,
        patch_size=patch_size,
        embed_dim=embed_dim,
        depth=depth,
        num_heads=num_heads,
        dropout=0.2,
        cross_attn_dim=cross_attn_dim
    )
    
    # Example forward pass
    x = torch.randn(n_samp, n_channels, dim1, dim2, dim3)
    features = model(x)
    
    print(f"Input shape: {x.shape}")
    print(f"Output features shape: {features.shape}")
    
    return model, features

# Example: create_3d_vit(8, 3, 32, 32, 32)

In [24]:
model, features = create_3d_vit(14, 12, 8, 8, 8)



Input shape: torch.Size([14, 12, 8, 8, 8])
Output features shape: torch.Size([14, 64, 256])


In [25]:
total_params = sum(p.numel() for p in model.parameters())
total_params





690358